# NUS ST4233 — Linear Models
## Student-intuitive case study with a real online dataset and Bokeh

**Dataset:** UCI Bike Sharing, daily observations (Capital Bikeshare, 2011–2012)  
**Response:** daily total rentals, `cnt`

This notebook is designed as a **teaching notebook**, not merely a collection of API calls.

For every major ST4233 idea we follow:

1. **Intuition** — what problem are we solving?
2. **Mathematics** — what exactly is being claimed?
3. **Implementation** — how do we compute it?
4. **Bokeh visualization / DataTable** — what does the result look like?
5. **Interpretation** — what should a student conclude?
6. **Pitfall** — what is commonly misunderstood?

The central theme is that regression, ANOVA, t-tests, F-tests, diagnostics, GLS and regularization are connected through the same mathematical object:

$$
\boxed{Y=X\beta+\varepsilon}
$$

## Learning map — how the notebook builds the subject

| Stage | ST4233 idea | Question we answer |
|---|---|---|
| 1 | Statistical design | What are the response, predictors, factors and leakage variables? |
| 2 | OLS | How is $\hat\beta$ obtained? |
| 3 | Projection geometry | Why are residuals orthogonal to the model space? |
| 4 | Random vectors | Where does $\operatorname{Var}(\hat\beta)$ come from? |
| 5 | Normal theory | Why do $\chi^2$, $t$ and $F$ distributions appear? |
| 6 | Gauss–Markov | What does BLUE actually mean? |
| 7 | General hypotheses | How do we unify coefficient tests, contrasts and partial F-tests? |
| 8 | ANOVA | Why is ANOVA regression with factor-coded columns? |
| 9 | Interactions | When does the effect of one predictor depend on another? |
| 10 | Multicollinearity | Why can coefficients become unstable even when predictions look fine? |
| 11 | Diagnostics | Which assumptions are questionable and what conclusions do they threaten? |
| 12 | Robust / GLS | What changes when variance or error correlation is not spherical? |
| 13 | Prediction | Why is predictive validation different from coefficient inference? |
| 14 | Ridge / GLM | How does classical linear-model theory connect to statistical learning? |

> **Recommended study habit:** after each section, explain the result in words before looking at the next formula.

## Case-study question

Imagine we are advising a bike-sharing operator.

We want to answer:

> **How do weather, season, calendar effects and time trend relate to daily bike demand, and how much confidence should we place in a classical linear-model explanation?**

This is useful for ST4233 because the dataset contains:

- a continuous response (`cnt`);
- continuous predictors (`temp`, `hum`, `windspeed`);
- categorical factors (`season`, `weathersit`, `weekday`);
- time structure (`dteday`, `yr`, `mnth`);
- correlated predictors (`temp` and `atemp`);
- a count response, which lets us discuss where ordinary Gaussian linear models stop being ideal.

### Important leakage warning

The dataset also contains:

$$
\texttt{cnt} = \texttt{casual} + \texttt{registered}.
$$

Therefore `casual` and `registered` **must never be used as predictors of `cnt`**.  
Doing so would be target leakage, not statistical learning.

In [1]:
# Core imports
import io
import zipfile
import urllib.request
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd

from scipy import stats
from scipy.linalg import svdvals

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_breusch_godfrey

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.layouts import gridplot, column, row
from bokeh.models import (
    ColumnDataSource, HoverTool, Span, DataTable, TableColumn,
    NumberFormatter, Div, FactorRange, LinearColorMapper, ColorBar, BasicTicker
)
from bokeh.transform import factor_cmap
from bokeh.palettes import Category10, RdBu11

warnings.filterwarnings("ignore")
output_notebook()

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("Environment ready.")

Loading BokehJS ...

Environment ready.


In [2]:
# ============================================================
# Reusable Bokeh teaching helpers
# ============================================================

def show_callout(title: str, body: str, width: int = 950):
    """Render a compact explanation box using Bokeh Div."""
    html = f"""
    <div style="border:1px solid #bbb;border-radius:10px;padding:14px 18px;
                background:#fafafa;font-family:Arial,sans-serif;line-height:1.5;">
      <div style="font-size:18px;font-weight:700;margin-bottom:6px;">{title}</div>
      <div style="font-size:14px;">{body}</div>
    </div>
    """
    show(Div(text=html, width=width))


def show_bokeh_table(
    frame: pd.DataFrame,
    title: str,
    explanation: str = "",
    width: int = 950,
    height: int = 280,
    digits: int = 4,
):
    """
    Render a pandas DataFrame as a Bokeh DataTable.

    Important Bokeh principle:
    once `source=ColumnDataSource(...)` is supplied, varying values used by
    glyphs/tables should be source fields rather than external Series/arrays.
    """
    df_view = frame.copy().reset_index(drop=True)
    source = ColumnDataSource(df_view)

    columns = []
    for col_name in df_view.columns:
        if pd.api.types.is_numeric_dtype(df_view[col_name]):
            formatter = NumberFormatter(format="0." + "0" * digits)
            columns.append(TableColumn(field=col_name, title=str(col_name), formatter=formatter))
        else:
            columns.append(TableColumn(field=col_name, title=str(col_name)))

    header = Div(
        text=(
            f"<h3 style='margin:0 0 5px 0'>{title}</h3>"
            f"<div style='font-size:13px;margin-bottom:8px'>{explanation}</div>"
        ),
        width=width,
    )
    table = DataTable(
        source=source,
        columns=columns,
        width=width,
        height=height,
        index_position=None,
        selectable=True,
    )
    show(column(header, table))


print("Bokeh teaching helpers ready.")

Bokeh teaching helpers ready.


## 1. Load the online UCI dataset

The official UCI repository identifies this as a regression dataset containing hourly and daily bike-rental counts with weather and seasonal information.

The notebook downloads the ZIP file directly from UCI and reads `day.csv`.

If the machine running the notebook is offline, download the UCI ZIP manually and place `day.csv` in the notebook's working directory.

In [3]:
UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"

def load_bike_daily_data(url: str = UCI_ZIP_URL) -> pd.DataFrame:
    # 1) Prefer a local copy if the user has already downloaded it.
    local_candidates = ["day.csv", "./data/day.csv"]
    for path in local_candidates:
        try:
            df = pd.read_csv(path)
            print(f"Loaded local dataset: {path}")
            return df
        except FileNotFoundError:
            pass

    # 2) Otherwise fetch the official UCI ZIP.
    try:
        with urllib.request.urlopen(url, timeout=30) as response:
            raw = response.read()

        with zipfile.ZipFile(io.BytesIO(raw)) as zf:
            with zf.open("day.csv") as f:
                df = pd.read_csv(f)

        print(f"Downloaded from UCI: {url}")
        return df

    except Exception as exc:
        raise RuntimeError(
            "Could not load the UCI Bike Sharing dataset.\n"
            "Either connect to the internet or download 'day.csv' from the "
            "UCI Bike Sharing dataset and place it beside this notebook."
        ) from exc

raw_df = load_bike_daily_data()
print("Shape:", raw_df.shape)
display(raw_df.head())

Downloaded from UCI: https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip
Shape: (731, 16)


,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.3442,0.3636,0.8058,0.1604,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.3635,0.3537,0.6961,0.2485,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.1964,0.1894,0.4373,0.2483,120,1229,1349
3,4,2011-01-04,1,0,1,0,2,1,1,0.2000,0.2121,0.5904,0.1603,108,1454,1562
4,5,2011-01-05,1,0,1,0,3,1,1,0.2270,0.2293,0.4370,0.1869,82,1518,1600


## 2. Data dictionary and modeling roles

A statistical model becomes much easier to reason about once every variable has an explicit role.

The most important distinction is:

- **response** — what we want to explain;
- **candidate predictor** — information available before the response;
- **identifier** — not a causal/predictive feature by itself;
- **leakage** — information that directly reveals the response.

In [4]:
variable_roles = pd.DataFrame([
    ("instant", "Identifier", "Row index; exclude from model."),
    ("dteday", "Time index", "Convert to datetime; useful for chronology/trend."),
    ("season", "Categorical predictor", "1=winter, 2=spring, 3=summer, 4=fall."),
    ("yr", "Categorical/time predictor", "0=2011, 1=2012; strong secular growth is plausible."),
    ("mnth", "Categorical predictor", "Month; overlaps strongly with season and time."),
    ("holiday", "Binary predictor", "Holiday indicator."),
    ("weekday", "Categorical predictor", "Day of week."),
    ("workingday", "Binary predictor", "Non-weekend and non-holiday indicator."),
    ("weathersit", "Categorical predictor", "Weather-condition category."),
    ("temp", "Continuous predictor", "Normalized measured temperature."),
    ("atemp", "Continuous predictor", "Normalized 'feels like' temperature."),
    ("hum", "Continuous predictor", "Normalized humidity."),
    ("windspeed", "Continuous predictor", "Normalized wind speed."),
    ("casual", "LEAKAGE", "Component of cnt; do not use to predict cnt."),
    ("registered", "LEAKAGE", "Component of cnt; do not use to predict cnt."),
    ("cnt", "Response", "Total daily rentals = casual + registered."),
], columns=["variable", "role", "reason"])

show_bokeh_table(
    variable_roles,
    title="Variable roles before modeling",
    explanation=("Statistical design comes before estimation. Integer-coded categories "
                 "are still categorical, and leakage variables must be excluded."),
    height=410
)

In [5]:
# Basic quality checks
quality = pd.DataFrame({
    "dtype": raw_df.dtypes.astype(str),
    "missing": raw_df.isna().sum(),
    "n_unique": raw_df.nunique(),
    "min": raw_df.select_dtypes(include=np.number).min(),
    "max": raw_df.select_dtypes(include=np.number).max(),
}).fillna("")

quality_view = quality.reset_index().rename(columns={"index": "variable"})
show_bokeh_table(
    quality_view,
    title="Data quality and support check",
    explanation=("Missingness is only one quality issue. We also inspect type, support, "
                 "cardinality, redundancy and target leakage."),
    height=390
)
print("Duplicate rows:", raw_df.duplicated().sum())
print("Leakage identity holds for all rows:",
      np.all(raw_df["casual"] + raw_df["registered"] == raw_df["cnt"]))

Duplicate rows: 0
Leakage identity holds for all rows: True


### Interpretation

For linear-model work, "clean data" does not merely mean "no missing values."

We also care about:

- whether predictors have sensible support;
- whether one variable deterministically contains another;
- whether categories should be treated as numeric or categorical;
- whether time ordering invalidates an independence assumption.

These are **model-design questions**, not just preprocessing questions.

In [6]:
# Create a teaching-friendly copy.
df = raw_df.copy()
df["dteday"] = pd.to_datetime(df["dteday"])
df = df.sort_values("dteday").reset_index(drop=True)

# Original Bike Sharing README coding: 1=spring, 2=summer, 3=fall, 4=winter.
season_map = {1: "Spring", 2: "Summer", 3: "Fall", 4: "Winter"}
weather_map = {1: "Clear/Partly cloudy", 2: "Mist/Cloudy", 3: "Light rain/snow", 4: "Heavy weather"}
weekday_map = {0: "Sun", 1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat"}

df["season_name"] = df["season"].map(season_map)
df["weather_name"] = df["weathersit"].map(weather_map)
df["weekday_name"] = df["weekday"].map(weekday_map)

# Keep the official normalized UCI features directly.
# This avoids introducing a unit conversion assumption into the statistical lesson.
df["temp_norm"] = df["temp"]
df["humidity_norm"] = df["hum"]
df["windspeed_norm"] = df["windspeed"]

# Continuous time trend in days from start.
df["time_index"] = (df["dteday"] - df["dteday"].min()).dt.days

# A centered temperature is useful when we later add interactions.
df["temp_centered"] = df["temp_norm"] - df["temp_norm"].mean()

display(df.head())

,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt,season_name,weather_name,weekday_name,temp_norm,humidity_norm,windspeed_norm,time_index,temp_centered
0,1,2011-01-01,1,0,1,0,6,0,2,0.3442,0.3636,0.8058,0.1604,331,654,985,Spring,Mist/Cloudy,Sat,0.3442,0.8058,0.1604,0,-0.1512
1,2,2011-01-02,1,0,1,0,0,0,2,0.3635,0.3537,0.6961,0.2485,131,670,801,Spring,Mist/Cloudy,Sun,0.3635,0.6961,0.2485,1,-0.1319
2,3,2011-01-03,1,0,1,0,1,1,1,0.1964,0.1894,0.4373,0.2483,120,1229,1349,Spring,Clear/Partly cloudy,Mon,0.1964,0.4373,0.2483,2,-0.2990
3,4,2011-01-04,1,0,1,0,2,1,1,0.2000,0.2121,0.5904,0.1603,108,1454,1562,Spring,Clear/Partly cloudy,Tue,0.2000,0.5904,0.1603,3,-0.2954
4,5,2011-01-05,1,0,1,0,3,1,1,0.2270,0.2293,0.4370,0.1869,82,1518,1600,Spring,Clear/Partly cloudy,Wed,0.2270,0.4370,0.1869,4,-0.2684


## 3. Exploratory data analysis: before fitting equations

EDA is not a substitute for inference, but it helps us identify which assumptions deserve attention.

We will inspect:

1. demand through time;
2. demand vs temperature;
3. demand by season;
4. predictor correlations.

The goal is to form hypotheses, not to declare significance from a plot.

In [7]:
# Bokeh: rentals through time
src = ColumnDataSource(df)

p_time = figure(
    width=900, height=330,
    x_axis_type="datetime",
    title="Daily bike rentals through time",
    x_axis_label="Date", y_axis_label="Daily rentals"
)
p_time.line("dteday", "cnt", source=src, line_width=2)
p_time.add_tools(HoverTool(
    tooltips=[("date", "@dteday{%F}"), ("rentals", "@cnt{0,0}")],
    formatters={"@dteday": "datetime"}
))

show(p_time)

In [8]:
# Bokeh: temperature vs rentals, with year represented as separate data sources.
p_scatter = figure(
    width=900, height=350,
    title="Temperature vs daily rentals",
    x_axis_label="Normalized temperature",
    y_axis_label="Daily rentals"
)

for year_value, label in [(0, "2011"), (1, "2012")]:
    sub = df[df["yr"] == year_value]
    p_scatter.scatter(
        x=sub["temp_norm"],
        y=sub["cnt"],
        size=7,
        alpha=0.45,
        legend_label=label
    )

p_scatter.legend.location = "top_left"
show(p_scatter)

In [9]:
# Table: seasonal descriptive statistics
season_summary = (
    df.groupby("season_name", observed=True)["cnt"]
      .agg(["count", "mean", "std", "median", "min", "max"])
      .reindex(["Spring", "Summer", "Fall", "Winter"])
)

season_summary_view = season_summary.reset_index()
season_summary_view["student_reading"] = [
    "Describe center/spread; do not infer significance yet."
] * len(season_summary_view)
show_bokeh_table(
    season_summary_view,
    title="Seasonal descriptive statistics",
    explanation=("ANOVA later asks whether between-season differences are large relative "
                 "to within-season variation."),
    height=260
)

print(
    "Reading the table:\n"
    "- 'mean' describes average demand.\n"
    "- 'std' measures within-season variability.\n"
    "- Differences in means are descriptive; ANOVA later asks whether they are "
    "large relative to within-season noise."
)

Reading the table:
- 'mean' describes average demand.
- 'std' measures within-season variability.
- Differences in means are descriptive; ANOVA later asks whether they are large relative to within-season noise.


In [10]:
# Bokeh box-plot-like visualization built from quartiles.
order = ["Spring", "Summer", "Fall", "Winter"]
g = df.groupby("season_name")["cnt"]
q1 = g.quantile(0.25).reindex(order)
q2 = g.quantile(0.50).reindex(order)
q3 = g.quantile(0.75).reindex(order)
lower = g.quantile(0.05).reindex(order)
upper = g.quantile(0.95).reindex(order)

box = pd.DataFrame({
    "season": order,
    "q1": q1.values,
    "median": q2.values,
    "q3": q3.values,
    "lower": lower.values,
    "upper": upper.values,
})

p_box = figure(
    x_range=order, width=900, height=350,
    title="Demand distribution by season",
    x_axis_label="Season", y_axis_label="Daily rentals"
)
p_box.segment("season", "upper", "season", "q3", source=ColumnDataSource(box), line_width=2)
p_box.segment("season", "lower", "season", "q1", source=ColumnDataSource(box), line_width=2)
p_box.vbar("season", 0.65, "q1", "q3", source=ColumnDataSource(box), alpha=0.35)
p_box.segment(
    x0=[x for x in order], y0=box["median"],
    x1=[x for x in order], y1=box["median"],
    line_width=4
)
show(p_box)

In [11]:
# ============================================================
# Correct Bokeh correlation heatmap
# ============================================================

corr_cols = ["cnt", "temp", "atemp", "hum", "windspeed", "time_index"]
corr = df[corr_cols].corr()

# Convert square matrix to tidy/long form.
corr_long = (
    corr.stack()
        .rename("corr")
        .reset_index()
        .rename(columns={"level_0": "x", "level_1": "y"})
)

# Put every varying visual/text property into the data source.
corr_long["label"] = corr_long["corr"].map(lambda value: f"{value:.2f}")
source_corr = ColumnDataSource(corr_long)

mapper = LinearColorMapper(
    palette=list(reversed(RdBu11)),
    low=-1,
    high=1,
)

p_corr = figure(
    x_range=corr_cols,
    y_range=list(reversed(corr_cols)),
    width=720,
    height=540,
    title="Pearson correlation matrix",
    toolbar_location="above",
)

p_corr.rect(
    x="x",
    y="y",
    width=1,
    height=1,
    source=source_corr,
    fill_color={"field": "corr", "transform": mapper},
    line_color="white",
)

p_corr.text(
    x="x",
    y="y",
    text="label",
    source=source_corr,
    text_align="center",
    text_baseline="middle",
    text_font_size="10pt",
)

p_corr.add_tools(
    HoverTool(
        tooltips=[
            ("pair", "@x × @y"),
            ("Pearson r", "@corr{0.000}"),
        ]
    )
)

p_corr.add_layout(
    ColorBar(color_mapper=mapper, ticker=BasicTicker(), title="Pearson r"),
    "right",
)

p_corr.xaxis.major_label_orientation = 0.8
p_corr.grid.grid_line_color = None
show(p_corr)

show_callout(
    "Why temp and atemp matter later",
    f"temp and atemp have correlation <b>{corr.loc['temp','atemp']:.3f}</b>. "
    "That near-redundancy becomes our real-data example of multicollinearity: "
    "prediction can remain reasonable while individual coefficients become unstable."
)

### What EDA suggests

Several themes should already be visible:

- demand increases substantially from 2011 to 2012;
- temperature and demand are positively related, but not perfectly linearly;
- season shifts both the center and spread of demand;
- `temp` and `atemp` are extremely correlated.

That last point will later produce a concrete demonstration of **multicollinearity**.

# Part I — The linear model as geometry

## 4. Start with a deliberately simple model

To see the mathematics cleanly, begin with:

$$
Y_i=\beta_0+\beta_1 T_i+\varepsilon_i,
$$

where $T_i$ is approximate temperature.

In matrix notation:

$$
\mathbf Y=X\beta+\varepsilon,
$$

with

$$
X=
\begin{bmatrix}
1&T_1\\
1&T_2\\
\vdots&\vdots\\
1&T_n
\end{bmatrix}.
$$

Ordinary least squares chooses $\hat\beta$ to minimize

$$
\|\mathbf Y-X\beta\|_2^2.
$$

In [12]:
y = df["cnt"].to_numpy(dtype=float)
X = np.column_stack([
    np.ones(len(df)),
    df["temp_norm"].to_numpy(dtype=float)
])

XtX = X.T @ X
Xty = X.T @ y
beta_hat_manual = np.linalg.solve(XtX, Xty)

simple_sm = sm.OLS(y, X).fit()

comparison = pd.DataFrame({
    "parameter": ["intercept", "temperature"],
    "manual_matrix_OLS": beta_hat_manual,
    "statsmodels_OLS": simple_sm.params,
    "absolute_difference": np.abs(beta_hat_manual - simple_sm.params)
})

comparison["student_message"] = "All methods should agree up to floating-point precision."
show_bokeh_table(
    comparison,
    title="Manual matrix OLS vs statsmodels",
    explanation=("The inverse formula is important for theory; numerical solvers/QR/SVD "
                 "are generally preferable in software."),
    height=210
)

### Why `solve` instead of an explicit matrix inverse?

The textbook formula is

$$
\hat\beta=(X^\top X)^{-1}X^\top Y.
$$

Numerically, computing the inverse explicitly is usually inferior to solving

$$
(X^\top X)\hat\beta=X^\top Y.
$$

For production numerical code, QR or SVD-based least squares is often even safer.

So:

- **formula:** inverse is useful for derivation;
- **implementation:** solve/QR/SVD is usually preferable.

In [13]:
# Compare three computational routes.
beta_solve = np.linalg.solve(X.T @ X, X.T @ y)
beta_lstsq = np.linalg.lstsq(X, y, rcond=None)[0]
beta_inv = np.linalg.inv(X.T @ X) @ X.T @ y

route_compare = pd.DataFrame({
    "solve_normal_equations": beta_solve,
    "least_squares_SVD": beta_lstsq,
    "explicit_inverse": beta_inv
}, index=["intercept", "temperature"]).reset_index().rename(columns={"index": "parameter"})
show_bokeh_table(
    route_compare,
    title="Three computational routes to OLS",
    explanation="Same mathematics, different numerical algorithms.",
    height=180
)

## 5. Hat matrix, projection and residual orthogonality

Define

$$
H=X(X^\top X)^{-1}X^\top.
$$

Then

$$
\hat Y=HY.
$$

The matrix $H$ is a projection onto the column space of $X$.

A projection matrix should satisfy:

$$
H^\top=H,\qquad H^2=H.
$$

The residual-maker matrix is

$$
M=I-H,
$$

so

$$
e=MY.
$$

A least-squares residual must be orthogonal to every column of $X$:

$$
X^\top e=0.
$$

In [14]:
# For pedagogy we compute H explicitly.
# For large n this is O(n^2) memory, so it should NOT be done casually in production.
H = X @ np.linalg.inv(X.T @ X) @ X.T
M = np.eye(len(X)) - H

y_hat = H @ y
resid = M @ y

projection_checks = pd.Series({
    "max |H - H.T|": np.max(np.abs(H - H.T)),
    "max |H@H - H|": np.max(np.abs(H @ H - H)),
    "max |M@M - M|": np.max(np.abs(M @ M - M)),
    "max |H@M|": np.max(np.abs(H @ M)),
    "max |X.T @ residual|": np.max(np.abs(X.T @ resid)),
    "rank(X)": np.linalg.matrix_rank(X),
    "trace(H)": np.trace(H),
    "rank(M)": np.linalg.matrix_rank(M),
})

projection_view = projection_checks.rename("value").reset_index().rename(columns={"index": "property"})
projection_view["expected"] = [
    "≈ 0", "≈ 0", "≈ 0", "≈ 0", "≈ 0", "= p", "= p", "= n-p"
]
show_bokeh_table(
    projection_view,
    title="Projection identities verified numerically",
    explanation=("These checks turn abstract properties such as symmetry, idempotence and "
                 "orthogonality into observable computations."),
    height=290
)

### Read the checks carefully

For a model with intercept + one slope, $p=2$.

Therefore:

$$
\operatorname{rank}(H)=\operatorname{tr}(H)=2,
$$

while

$$
\operatorname{rank}(M)=n-2.
$$

That is the geometric origin of **residual degrees of freedom**.

The tiny nonzero numerical values in identities such as $H^2-H$ are floating-point error, not theory failure.

In [15]:
# SST = SSR + SSE, derived from orthogonality.
y_bar = y.mean()
SST = np.sum((y - y_bar) ** 2)
SSR = np.sum((y_hat - y_bar) ** 2)
SSE = np.sum((y - y_hat) ** 2)

ss_table = pd.DataFrame({
    "quantity": ["SST", "SSR", "SSE", "SSR + SSE", "decomposition error"],
    "value": [SST, SSR, SSE, SSR + SSE, SST - (SSR + SSE)],
    "meaning": [
        "Total variation around the sample mean",
        "Variation explained by the fitted regression",
        "Residual/unexplained variation",
        "Explained + residual",
        "Should be ~0 when an intercept is present"
    ]
})

show_bokeh_table(
    ss_table,
    title="Sum-of-squares decomposition",
    explanation=("With an intercept, total centered variation decomposes into orthogonal "
                 "explained and residual components."),
    height=220
)

## 6. Visualize the projection in the familiar 2D regression picture

The true projection lives in $\mathbb R^n$, not on the page.

But the ordinary regression scatterplot provides a useful shadow of the geometry:

- vertical positions are observed $Y_i$;
- the line gives fitted $\hat Y_i$;
- vertical gaps are residuals.

The deeper $n$-dimensional statement is that the full residual vector is orthogonal to the model subspace.

In [16]:
plot_df = df[["temp_norm", "cnt"]].copy().sort_values("temp_norm")
X_plot = np.column_stack([np.ones(len(plot_df)), plot_df["temp_norm"]])
plot_df["fitted"] = X_plot @ beta_hat_manual

p_fit = figure(
    width=900, height=380,
    title="Simple OLS fit: demand vs temperature",
    x_axis_label="Normalized temperature",
    y_axis_label="Daily rentals"
)
p_fit.scatter(plot_df["temp_norm"], plot_df["cnt"], size=6, alpha=0.35)
p_fit.line(plot_df["temp_norm"], plot_df["fitted"], line_width=3)
show(p_fit)

# Part II — Probability behind regression inference

## 7. Random vectors and covariance propagation

Suppose

$$
Y=X\beta+\varepsilon,\qquad
E(\varepsilon)=0,\qquad
\operatorname{Var}(\varepsilon)=\sigma^2I.
$$

Because

$$
\hat\beta=AY,\qquad
A=(X^\top X)^{-1}X^\top,
$$

we use the random-vector identity

$$
\operatorname{Var}(AY)=A\operatorname{Var}(Y)A^\top.
$$

Therefore

$$
\operatorname{Var}(\hat\beta)
=
\sigma^2(X^\top X)^{-1}.
$$

This formula links **design geometry** to **estimation uncertainty**.

In [17]:
sigma2_hat = SSE / (len(y) - X.shape[1])
cov_beta_manual = sigma2_hat * np.linalg.inv(X.T @ X)

cov_comparison = pd.DataFrame(
    cov_beta_manual,
    index=["intercept", "temperature"],
    columns=["intercept", "temperature"]
)

show_bokeh_table(
    cov_comparison.reset_index().rename(columns={"index": "row_parameter"}),
    title="Estimated covariance matrix of β-hat",
    explanation="Diagonal entries are coefficient variances; their square roots are standard errors.",
    height=200
)

print("Manual SEs:", np.sqrt(np.diag(cov_beta_manual)))
print("statsmodels SEs:", simple_sm.bse)

Manual SEs: [161.16353106 305.18803094]
statsmodels SEs: [161.16353106 305.18803094]


## 8. Multivariate normality and the sampling distribution of $\hat\beta$

Under the stronger Gaussian assumption

$$
\varepsilon\sim N(0,\sigma^2I),
$$

we obtain

$$
\hat\beta
\sim
N\left(
\beta,\,
\sigma^2(X^\top X)^{-1}
\right).
$$

This is not just an abstract theorem. We can simulate repeated datasets while holding $X$ fixed and watch the estimator vary.

In [18]:
# Controlled simulation: same X, known beta and sigma.
# Scale x for numerical readability.
x_sim = np.linspace(-2, 2, 120)
X_sim = np.column_stack([np.ones_like(x_sim), x_sim])
beta_true = np.array([100.0, 25.0])
sigma_true = 30.0
n_rep = 4000

B = np.linalg.inv(X_sim.T @ X_sim) @ X_sim.T

eps = rng.normal(0, sigma_true, size=(len(x_sim), n_rep))
Y_sim = X_sim @ beta_true[:, None] + eps
beta_sims = (B @ Y_sim).T

empirical = pd.DataFrame({
    "quantity": [
        "E[intercept hat]", "true intercept",
        "E[slope hat]", "true slope",
        "Var(intercept hat)", "theoretical Var(intercept hat)",
        "Var(slope hat)", "theoretical Var(slope hat)"
    ],
    "value": [
        beta_sims[:, 0].mean(), beta_true[0],
        beta_sims[:, 1].mean(), beta_true[1],
        beta_sims[:, 0].var(ddof=1),
        (sigma_true**2 * np.linalg.inv(X_sim.T @ X_sim))[0, 0],
        beta_sims[:, 1].var(ddof=1),
        (sigma_true**2 * np.linalg.inv(X_sim.T @ X_sim))[1, 1]
    ]
})
show_bokeh_table(
    empirical,
    title="Monte Carlo check of OLS sampling theory",
    explanation=("Empirical means/variances across repeated samples should approach the "
                 "theoretical values implied by the linear model."),
    height=260
)

In [19]:
hist, edges = np.histogram(beta_sims[:, 1], bins=45, density=True)
centers = (edges[:-1] + edges[1:]) / 2

theory_var_slope = (sigma_true**2 * np.linalg.inv(X_sim.T @ X_sim))[1, 1]
grid = np.linspace(edges.min(), edges.max(), 300)
pdf = stats.norm.pdf(grid, loc=beta_true[1], scale=np.sqrt(theory_var_slope))

p_sampling = figure(
    width=900, height=350,
    title="Sampling distribution of the OLS slope",
    x_axis_label="Estimated slope", y_axis_label="Density"
)
p_sampling.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:], alpha=0.35)
p_sampling.line(grid, pdf, line_width=3, legend_label="Theoretical Normal density")
p_sampling.legend.location = "top_left"
show(p_sampling)

## 9. Quadratic forms and why SSE becomes chi-square

The residual sum of squares is

$$
SSE=e^\top e.
$$

Since $e=MY$,

$$
SSE=Y^\top MY.
$$

That is a **quadratic form**.

Under a correct Gaussian linear model,

$$
\frac{SSE}{\sigma^2}\sim\chi^2_{n-p}.
$$

This result explains why

$$
s^2=\frac{SSE}{n-p}
$$

is unbiased for $\sigma^2$, and it is one ingredient behind $t$- and $F$-statistics.

In [20]:
# Verify the quadratic-form identity on the real simple-regression fit.
quadratic_sse = y.T @ M @ y
direct_sse = resid.T @ resid

qf_identity = pd.DataFrame({
    "method": ["y.T @ M @ y", "residual.T @ residual"],
    "SSE": [quadratic_sse, direct_sse],
    "meaning": ["Quadratic-form view", "Direct residual sum of squares"]
})
show_bokeh_table(
    qf_identity,
    title="SSE is a quadratic form",
    explanation="The two calculations should agree numerically.",
    height=170
)

In [21]:
# Monte Carlo demonstration of SSE/sigma^2 ~ chi-square_{n-p}
n_q = 50
p_q = 3
x1 = np.linspace(-1, 1, n_q)
x2 = rng.normal(size=n_q)
X_q = np.column_stack([np.ones(n_q), x1, x2])
H_q = X_q @ np.linalg.inv(X_q.T @ X_q) @ X_q.T
M_q = np.eye(n_q) - H_q

sigma_q = 2.0
reps_q = 3000
eps_q = rng.normal(0, sigma_q, size=(n_q, reps_q))
q_values = np.einsum("ir,ij,jr->r", eps_q, M_q, eps_q) / sigma_q**2

df_q = n_q - p_q

hist_q, edges_q = np.histogram(q_values, bins=45, density=True)
grid_q = np.linspace(0, np.quantile(q_values, 0.995), 300)

p_qf = figure(
    width=900, height=350,
    title=f"Quadratic-form result: SSE/σ² vs χ²({df_q})",
    x_axis_label="SSE / σ²", y_axis_label="Density"
)
p_qf.quad(top=hist_q, bottom=0, left=edges_q[:-1], right=edges_q[1:], alpha=0.35)
p_qf.line(grid_q, stats.chi2.pdf(grid_q, df_q), line_width=3)
show(p_qf)

chi_check = pd.DataFrame({
    "quantity": ["Empirical mean", "Chi-square theoretical mean",
                 "Empirical variance", "Chi-square theoretical variance"],
    "value": [q_values.mean(), df_q, q_values.var(ddof=1), 2*df_q]
})
show_bokeh_table(
    chi_check,
    title=f"Chi-square quadratic-form check (df={df_q})",
    explanation="For χ²ν, mean=ν and variance=2ν.",
    height=190
)

# Part III — Gauss–Markov and estimation quality

## 10. What does BLUE really mean?

Under

$$
E(\varepsilon)=0,\qquad
\operatorname{Var}(\varepsilon)=\sigma^2I,
$$

OLS is the **Best Linear Unbiased Estimator**.

"Best" does **not** mean:

- best predictive algorithm among all imaginable algorithms;
- minimum MSE among biased estimators;
- robust to arbitrary assumption violations.

It means:

> among **linear unbiased** estimators of $\beta$, OLS has minimum covariance.

We can compare OLS to an intentionally arbitrary weighted least-squares estimator.  
With deterministic positive weights, WLS remains linear and unbiased when the mean model is correct, but under truly homoskedastic errors its variance should not beat OLS.

In [22]:
# Gauss-Markov simulation
n_gm = 100
x_gm = np.linspace(-2, 2, n_gm)
X_gm = np.column_stack([np.ones(n_gm), x_gm])
beta_gm = np.array([3.0, 2.0])
sigma_gm = 1.5

# Arbitrary deterministic weights, not justified by the true error variance.
w = np.linspace(0.3, 3.0, n_gm)
W = np.diag(w)

A_ols = np.linalg.inv(X_gm.T @ X_gm) @ X_gm.T
A_wls = np.linalg.inv(X_gm.T @ W @ X_gm) @ X_gm.T @ W

reps_gm = 5000
eps_gm = rng.normal(0, sigma_gm, size=(n_gm, reps_gm))
Y_gm = X_gm @ beta_gm[:, None] + eps_gm

b_ols = (A_ols @ Y_gm).T
b_wls = (A_wls @ Y_gm).T

gm_results = pd.DataFrame({
    "estimator": ["OLS", "Arbitrary WLS"],
    "mean_intercept": [b_ols[:,0].mean(), b_wls[:,0].mean()],
    "var_intercept": [b_ols[:,0].var(ddof=1), b_wls[:,0].var(ddof=1)],
    "mean_slope": [b_ols[:,1].mean(), b_wls[:,1].mean()],
    "var_slope": [b_ols[:,1].var(ddof=1), b_wls[:,1].var(ddof=1)],
})

gm_results["interpretation"] = [
    "BLUE reference under spherical errors.",
    "Arbitrary weights generally increase variance when homoskedasticity is actually correct."
]
show_bokeh_table(
    gm_results,
    title="Gauss–Markov simulation: OLS vs arbitrary WLS",
    explanation=("Both estimators can be unbiased, but 'best' means OLS has minimum covariance "
                 "within the class of linear unbiased estimators."),
    height=210
)
print("True beta:", beta_gm)

True beta: [3. 2.]


### Interpretation

Both estimators should be centered near the true coefficients.

But the arbitrary WLS estimator generally has greater variance because the weights add distortion without matching any genuine heteroskedasticity.

That is the operational meaning of Gauss–Markov in this experiment.

Later we will reverse the situation: if errors are **not** spherical and their covariance structure is known/estimable, GLS can improve efficiency.

# Part IV — A richer demand model

## 11. Inferential model specification

Now move from the toy simple regression to a realistic model.

We use:

$$
\text{cnt}
\sim
\text{temperature}
+
\text{humidity}
+
\text{windspeed}
+
\text{year}
+
\text{season}
+
\text{weather}
+
\text{workingday}
+
\text{time trend}.
$$

### Why not include everything?

Good linear modeling requires **design discipline**.

We intentionally omit:

- `casual`, `registered`: target leakage;
- `atemp`: nearly duplicates temperature, useful later for a collinearity demonstration;
- `mnth`: heavily overlaps season + time trend;
- raw `instant`: identifier;
- both `weekday` and `workingday` initially: redundant calendar encoding may complicate interpretation.

This is a modeling decision, not a software limitation.

## Before fitting the richer model: distinguish *estimand* from *algorithm*

The most important modeling question is not "which library function should I call?"

It is:

> **What conditional comparison do I want each coefficient to represent?**

A multiple-regression coefficient is a *partial* or *conditional* association. For a continuous predictor $X_j$, it describes the modeled change in the mean response for a one-unit change in $X_j$, **holding the other encoded columns fixed**.

That conditional phrase is why feature redundancy, interactions and omitted variables matter so much.

In [23]:
formula_base = (
    "cnt ~ temp_norm + humidity_norm + windspeed_norm "
    "+ C(yr) + C(season_name) + C(weather_name) + workingday + time_index"
)

model_base = smf.ols(formula_base, data=df).fit()
print(model_base.summary())

                            OLS Regression Results                            
Dep. Variable:                    cnt   R-squared:                       0.820
Model:                            OLS   Adj. R-squared:                  0.817
Method:                 Least Squares   F-statistic:                     298.3
Date:                Sun, 06 Sep 2026   Prob (F-statistic):          2.93e-259
Time:                        00:39:43   Log-Likelihood:                -5942.4
No. Observations:                 731   AIC:                         1.191e+04
Df Residuals:                     719   BIC:                         1.196e+04
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                                         coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Inte

## 12. Turn the coefficient table into an interpretation table

A regression table is useful only if we know what each row means.

For numerical predictors, coefficients describe a change per one-unit increase **holding other modeled predictors fixed**.

For categorical predictors, coefficients compare a level with a reference category.

In [24]:
coef_table = pd.DataFrame({
    "estimate": model_base.params,
    "std_error": model_base.bse,
    "t_stat": model_base.tvalues,
    "p_value": model_base.pvalues,
    "ci_low": model_base.conf_int()[0],
    "ci_high": model_base.conf_int()[1],
})

coef_table["signal_to_se"] = np.abs(coef_table["estimate"] / coef_table["std_error"])
coef_view = coef_table.sort_values("p_value").reset_index().rename(columns={"index": "term"})
coef_view["student_interpretation"] = np.where(
    coef_view["p_value"] < 0.05,
    "Evidence against a zero conditional coefficient under the stated model assumptions.",
    "No strong evidence against zero at the 5% level; this is not proof of no effect."
)
show_bokeh_table(
    coef_view,
    title="Coefficient estimates with uncertainty",
    explanation=("Read each coefficient conditionally on the other columns in X. "
                 "Statistical significance is not automatically causal significance."),
    height=430
)

### Interpretation checklist for every coefficient

Before saying "X affects Y", ask:

1. **What is the reference group?**
2. **What other predictors are held fixed?**
3. **What is the unit of X?**
4. **Is the relationship plausibly linear over the observed range?**
5. **Does observational association justify causal language?** Usually no.
6. **Are the standard errors trustworthy under heteroskedasticity/autocorrelation?**

ST4233 is not only about calculating $\hat\beta$; it is about knowing when the inferential statement attached to $\hat\beta$ is justified.

In [25]:
model_quality = pd.DataFrame({
    "metric": ["n", "parameters", "residual_df", "R2", "adjusted_R2", "AIC", "BIC", "RMSE_in_sample"],
    "value": [
        model_base.nobs,
        len(model_base.params),
        model_base.df_resid,
        model_base.rsquared,
        model_base.rsquared_adj,
        model_base.aic,
        model_base.bic,
        np.sqrt(np.mean(model_base.resid**2))
    ],
    "interpretation": [
        "Number of observations used",
        "Number of fitted coefficients",
        "Dimensions left for residual variation",
        "Fraction of sample variation explained",
        "R² penalized for model size",
        "Likelihood-based criterion; lower is better among comparable models",
        "Stronger complexity penalty than AIC",
        "Typical in-sample residual magnitude"
    ]
})
show_bokeh_table(
    model_quality,
    title="Model-level fit summary",
    explanation=("R² describes in-sample explained variation; it does not validate assumptions "
                 "or guarantee future predictive performance."),
    height=270
)

# Part V — Confidence intervals, prediction intervals and general hypotheses

## 13. Mean-response CI vs individual prediction interval

For a new design vector $x_0$,

$$
\hat y_0=x_0^\top\hat\beta.
$$

A **confidence interval for the mean response** quantifies uncertainty in:

$$
E(Y\mid x_0).
$$

A **prediction interval** additionally includes the new observation's irreducible noise:

$$
Y_0=x_0^\top\beta+\varepsilon_0.
$$

Therefore prediction intervals are wider.

In [26]:
# Create a temperature profile while holding other variables at representative values.
temp_grid = np.linspace(df["temp_norm"].quantile(0.05),
                        df["temp_norm"].quantile(0.95), 80)

profile = pd.DataFrame({
    "temp_norm": temp_grid,
    "humidity_norm": df["humidity_norm"].median(),
    "windspeed_norm": df["windspeed_norm"].median(),
    "yr": 1,
    "season_name": "Summer",
    "weather_name": "Clear/Partly cloudy",
    "workingday": 1,
    "time_index": df["time_index"].median()
})

pred_frame = model_base.get_prediction(profile).summary_frame(alpha=0.05)
plot_pred = profile.copy()
plot_pred = pd.concat([plot_pred.reset_index(drop=True), pred_frame.reset_index(drop=True)], axis=1)

display(plot_pred.head())

,temp_norm,humidity_norm,windspeed_norm,yr,season_name,weather_name,workingday,time_index,mean,mean_se,mean_ci_lower,mean_ci_upper,obs_ci_lower,obs_ci_upper
0,0.2136,0.6267,0.1810,1,Summer,Clear/Partly cloudy,1,365.0000,"4,701.0615",152.9015,"4,400.8748","5,001.2482","3,048.7948","6,353.3282"
1,0.2206,0.6267,0.1810,1,Summer,Clear/Partly cloudy,1,365.0000,"4,736.8164",151.3161,"4,439.7423","5,033.8906","3,085.1124","6,388.5205"
2,0.2276,0.6267,0.1810,1,Summer,Clear/Partly cloudy,1,365.0000,"4,772.5713",149.7454,"4,478.5809","5,066.5617","3,121.4192","6,423.7235"
3,0.2347,0.6267,0.1810,1,Summer,Clear/Partly cloudy,1,365.0000,"4,808.3263",148.1898,"4,517.3899","5,099.2627","3,157.7151","6,458.9374"
4,0.2417,0.6267,0.1810,1,Summer,Clear/Partly cloudy,1,365.0000,"4,844.0812",146.6499,"4,556.1680","5,131.9943","3,194.0002","6,494.1621"


In [27]:
p_pi = figure(
    width=900, height=400,
    title="Mean-response confidence band vs individual prediction band",
    x_axis_label="Normalized temperature",
    y_axis_label="Predicted daily rentals"
)

p_pi.varea(
    x=plot_pred["temp_norm"],
    y1=plot_pred["obs_ci_lower"],
    y2=plot_pred["obs_ci_upper"],
    alpha=0.15,
    legend_label="95% prediction interval"
)
p_pi.varea(
    x=plot_pred["temp_norm"],
    y1=plot_pred["mean_ci_lower"],
    y2=plot_pred["mean_ci_upper"],
    alpha=0.30,
    legend_label="95% CI for mean"
)
p_pi.line(
    plot_pred["temp_norm"],
    plot_pred["mean"],
    line_width=3,
    legend_label="Predicted mean"
)
p_pi.legend.location = "top_left"
show(p_pi)

## 14. General linear hypotheses: $H_0:C\beta=d$

Many apparently different tests are one matrix idea:

$$
H_0:C\beta=d.
$$

Examples:

- one coefficient equals zero;
- several coefficients jointly equal zero;
- two coefficients are equal;
- a linear combination equals a specified value.

We will test whether the **season factor is jointly unnecessary**, conditional on the rest of the model.

In [28]:
# Reduced model removes season.
formula_no_season = (
    "cnt ~ temp_norm + humidity_norm + windspeed_norm "
    "+ C(yr) + C(weather_name) + workingday + time_index"
)
model_no_season = smf.ols(formula_no_season, data=df).fit()

nested_season_test = anova_lm(model_no_season, model_base)
show_bokeh_table(
    nested_season_test.reset_index().rename(columns={"index": "model"}),
    title="Nested-model F-test for the season factor",
    explanation=("The test compares the extra signal contributed by season directions with "
                 "the residual noise of the full model."),
    height=190
)

print(
    "Interpretation: the F-test compares the extra variation explained by the "
    "season parameters with the residual noise level of the full model."
)

Interpretation: the F-test compares the extra variation explained by the season parameters with the residual noise level of the full model.


In [29]:
# Direct matrix-style joint restriction using parameter names.
season_terms = [name for name in model_base.params.index if "C(season_name)" in name]
restriction_text = ", ".join(f"{term} = 0" for term in season_terms)

print("Joint restriction:")
print(restriction_text)

season_f = model_base.f_test(restriction_text)
print(season_f)

Joint restriction:
C(season_name)[T.Spring] = 0, C(season_name)[T.Summer] = 0, C(season_name)[T.Winter] = 0
<F test: F=65.32823532076095, p=2.3126875330707972e-37, df_denom=719, df_num=3>


### The two tests should agree

`anova_lm(reduced, full)` and `full.f_test(...)` are two computational views of the same general linear-hypothesis principle.

This is exactly why the matrix formulation is powerful: individual t-tests, ANOVA comparisons and nested-model F-tests are not disconnected procedures.

# Part VI — ANOVA is regression with dummy variables

## 15. One-way ANOVA: season only

A traditional one-way ANOVA can be written:

$$
Y_{ij}=\mu+\alpha_i+\varepsilon_{ij}.
$$

A regression implementation writes the same information using indicator variables.

The null hypothesis is:

$$
H_0:\mu_{\text{Winter}}=
\mu_{\text{Spring}}=
\mu_{\text{Summer}}=
\mu_{\text{Fall}}.
$$

In [30]:
season_anova_model = smf.ols("cnt ~ C(season_name)", data=df).fit()
season_anova_table = anova_lm(season_anova_model, typ=2)

show_bokeh_table(
    season_anova_table.reset_index().rename(columns={"index": "source"}),
    title="One-way ANOVA table: season",
    explanation="ANOVA is regression with categorical design columns.",
    height=180
)

In [31]:
# Verify that fitted group means equal observed group means.
means_observed = df.groupby("season_name")["cnt"].mean().sort_index()

pred_by_season = (
    df.assign(fitted=season_anova_model.fittedvalues)
      .groupby("season_name")["fitted"]
      .mean()
      .sort_index()
)

anova_mean_check = pd.DataFrame({
    "observed_group_mean": means_observed,
    "regression_fitted_group_mean": pred_by_season,
    "difference": means_observed - pred_by_season
})

anova_mean_view = anova_mean_check.reset_index()
anova_mean_view["student_message"] = "One-factor ANOVA fitted values reproduce group means."
show_bokeh_table(
    anova_mean_view,
    title="ANOVA-as-regression verification",
    explanation="Within each season, the one-factor fitted value is the season mean.",
    height=210
)

### Important caution

A one-way season ANOVA answers a **marginal** question:

> are average rentals different across seasons?

It does not control for:

- year growth;
- weather;
- temperature;
- time trend.

The multiple-regression model asks a more conditional question.

This distinction between **marginal** and **adjusted/conditional** comparisons is crucial in statistical modeling.

# Part VII — Interactions

## 16. Does the temperature slope depend on season?

A model without interaction assumes the same temperature slope in every season.

An interaction model allows:

$$
\text{temperature effect}
=
\text{baseline slope}
+
\text{season-specific slope adjustment}.
$$

Formally:

$$
Y
=
\beta_0+\beta_1T+\alpha_s+\gamma_sT+\cdots+\varepsilon.
$$

If $\gamma_s\neq0$, the effect of temperature depends on season.

In [52]:
formula_interaction = (
    "cnt ~ temp_centered * C(season_name) "
    "+ humidity_norm + windspeed_norm + C(yr) + C(weather_name) "
    "+ workingday + time_index"
)

model_interaction = smf.ols(formula_interaction, data=df).fit()
interaction_test = anova_lm(model_base, model_interaction)

show_bokeh_table(
    interaction_test.reset_index().rename(columns={"index": "model"}),
    title="Nested comparison: common slope vs temperature × season interaction",
    explanation="The interaction asks whether the temperature relationship changes by season.",
    height=190
)

### Why center temperature before the interaction?

Without centering, the "main effect" of season is evaluated at $T=0^\circ C$, which may be an awkward comparison point.

With

$$
T_c=T-\bar T,
$$

the season main effects are interpreted around the **average observed temperature**.

Centering does not remove the interaction; it improves interpretability and sometimes numerical conditioning.

In [53]:
# Visualize fitted season-specific temperature slopes, holding other variables fixed.
seasons = ["Spring", "Summer", "Fall", "Winter"]
temp_grid2 = np.linspace(df["temp_norm"].quantile(0.05),
                         df["temp_norm"].quantile(0.95), 90)

p_int = figure(
    width=900, height=420,
    title="Interaction model: fitted temperature relationship by season",
    x_axis_label="Normalized temperature",
    y_axis_label="Predicted rentals"
)

for s in seasons:
    g = pd.DataFrame({
        "temp_centered": temp_grid2 - df["temp_norm"].mean(),
        "season_name": s,
        "humidity_norm": df["humidity_norm"].median(),
        "windspeed_norm": df["windspeed_norm"].median(),
        "yr": 1,
        "weather_name": "Clear/Partly cloudy",
        "workingday": 1,
        "time_index": df["time_index"].median()
    })
    y_pred = model_interaction.predict(g)
    p_int.line(temp_grid2, y_pred, line_width=2.5, legend_label=s)

p_int.legend.location = "top_left"
show(p_int)

# Part VIII — Multicollinearity and conditioning

## 17. Deliberately create a collinearity problem

The UCI data contains both:

- `temp`: measured temperature;
- `atemp`: "feels like" temperature.

They are extremely correlated.

If both are entered into a model, the model may predict reasonably well while individual coefficient estimates become unstable.

This is a classic reminder:

$$
\text{prediction stability} \neq \text{coefficient stability}.
$$

In [54]:
corr_temp_atemp = df[["temp", "atemp"]].corr().iloc[0, 1]
print("corr(temp, atemp) =", corr_temp_atemp)

model_temp_only = smf.ols("cnt ~ temp + C(yr) + time_index", data=df).fit()
model_temp_atemp = smf.ols("cnt ~ temp + atemp + C(yr) + time_index", data=df).fit()

compare_collinear = pd.DataFrame({
    "model": ["temp only", "temp + atemp"],
    "R2": [model_temp_only.rsquared, model_temp_atemp.rsquared],
    "adj_R2": [model_temp_only.rsquared_adj, model_temp_atemp.rsquared_adj],
    "temp_coef": [model_temp_only.params.get("temp", np.nan),
                  model_temp_atemp.params.get("temp", np.nan)],
    "temp_SE": [model_temp_only.bse.get("temp", np.nan),
                model_temp_atemp.bse.get("temp", np.nan)],
    "atemp_coef": [np.nan, model_temp_atemp.params.get("atemp", np.nan)],
    "atemp_SE": [np.nan, model_temp_atemp.bse.get("atemp", np.nan)]
})

compare_collinear["student_message"] = [
    "Baseline: one temperature representation.",
    "Near-duplicate predictors can destabilize individual coefficients without much R² gain."
]
show_bokeh_table(
    compare_collinear,
    title=f"Collinearity experiment: corr(temp, atemp)={corr_temp_atemp:.3f}",
    explanation="Coefficient stability and predictive fit are different properties.",
    height=210
)

corr(temp, atemp) = 0.9917015532294647


In [55]:
# VIF calculation for a simple numerical design.
vif_X = df[["temp", "atemp", "hum", "windspeed", "time_index"]].copy()
vif_X = sm.add_constant(vif_X)

vif_table = pd.DataFrame({
    "feature": vif_X.columns,
    "VIF": [
        variance_inflation_factor(vif_X.to_numpy(), i)
        for i in range(vif_X.shape[1])
    ]
})
vif_table["student_interpretation"] = np.where(
    vif_table["feature"] == "const",
    "Intercept VIF is usually not interpreted.",
    np.where(vif_table["VIF"] >= 10, "Severe redundancy warning",
             np.where(vif_table["VIF"] >= 5, "Moderate/strong redundancy warning", "No major VIF warning"))
)
show_bokeh_table(
    vif_table,
    title="Variance Inflation Factors",
    explanation="VIF quantifies coefficient-variance inflation caused by linear redundancy.",
    height=230
)

In [56]:
# Condition-number/eigenvalue view.
# Standardize non-constant predictors so scale differences do not dominate.
Z = StandardScaler().fit_transform(df[["temp", "atemp", "hum", "windspeed", "time_index"]])
singular_values = svdvals(Z)
condition_number = singular_values.max() / singular_values.min()

conditioning = pd.DataFrame({
    "singular_value": singular_values,
    "inverse_squared_scale": 1 / (singular_values**2)
})

show_bokeh_table(
    conditioning,
    title=f"Singular-value view | condition number={condition_number:.2f}",
    explanation=("Small singular values correspond to weakly identified design directions; "
                 "inverse scaling magnifies coefficient uncertainty."),
    height=220
)
print("Condition number of standardized design:", condition_number)

Condition number of standardized design: 16.463986209634157


### Why does collinearity inflate variance?

If

$$
X^\top X=Q\Lambda Q^\top,
$$

then

$$
(X^\top X)^{-1}=Q\Lambda^{-1}Q^\top.
$$

A small eigenvalue $\lambda_j$ produces a large $1/\lambda_j$.

Because

$$
\operatorname{Var}(\hat\beta)=\sigma^2(X^\top X)^{-1},
$$

uncertainty explodes in poorly identified directions.

This is a much deeper explanation than simply saying "the predictors are correlated."

# Part IX — Diagnostics and assumption checking

## 18. Residual diagnostics

The classical model assumes approximately:

1. correct linear mean structure;
2. zero-mean errors conditional on predictors;
3. constant conditional variance;
4. independent errors;
5. Gaussian errors for exact small-sample $t/F$ inference.

Real data rarely obey every assumption perfectly.

The right response is not to panic; it is to diagnose **which inferential statement is threatened**.

## A diagnostic is not a pass/fail exam

Real datasets almost never satisfy every classical assumption perfectly.

For each diagnostic, ask three questions:

1. **What assumption is being examined?**
2. **Which result depends on that assumption?** Point estimate? Standard error? Exact p-value? Prediction?
3. **What is the least disruptive appropriate response?** Better mean specification, robust covariance, GLS, transformation, sensitivity analysis, or a different response distribution?

This is much more useful than mechanically searching for a p-value above 0.05.

In [57]:
influence = model_base.get_influence()
diag = pd.DataFrame({
    "date": df["dteday"],
    "observed": df["cnt"],
    "fitted": model_base.fittedvalues,
    "residual": model_base.resid,
    "studentized_internal": influence.resid_studentized_internal,
    "studentized_external": influence.resid_studentized_external,
    "leverage": influence.hat_matrix_diag,
    "cooks_d": influence.cooks_distance[0],
})

top_influence = diag.sort_values("cooks_d", ascending=False).head(12).copy()
top_influence["student_message"] = "Influential ≠ automatically erroneous; inspect before acting."
show_bokeh_table(
    top_influence,
    title="Most influential observations",
    explanation="Cook's distance measures sensitivity of the fitted model to an observation.",
    height=330
)

In [38]:
p_resid = figure(
    width=900, height=360,
    title="Residuals vs fitted values",
    x_axis_label="Fitted rentals",
    y_axis_label="Residual"
)
p_resid.scatter(diag["fitted"], diag["residual"], size=6, alpha=0.45)
p_resid.add_layout(Span(location=0, dimension="width", line_dash="dashed"))

show(p_resid)

In [58]:
# Normal Q-Q data, visualized with Bokeh.
resid_sorted = np.sort(model_base.resid)
n = len(resid_sorted)
theoretical_q = stats.norm.ppf((np.arange(1, n + 1) - 0.5) / n)

# Fit a reference line through quartiles.
q_theory = np.quantile(theoretical_q, [0.25, 0.75])
q_resid = np.quantile(resid_sorted, [0.25, 0.75])
qq_slope = (q_resid[1] - q_resid[0]) / (q_theory[1] - q_theory[0])
qq_intercept = q_resid[0] - qq_slope * q_theory[0]

p_qq = figure(
    width=900, height=360,
    title="Normal Q-Q diagnostic",
    x_axis_label="Theoretical Normal quantile",
    y_axis_label="Observed residual quantile"
)
p_qq.scatter(theoretical_q, resid_sorted, size=6, alpha=0.45)
p_qq.line(
    [theoretical_q.min(), theoretical_q.max()],
    [
        qq_intercept + qq_slope * theoretical_q.min(),
        qq_intercept + qq_slope * theoretical_q.max()
    ],
    line_width=2
)
show(p_qq)

In [59]:
# Residuals through time: useful for spotting autocorrelation / omitted temporal structure.
p_rt = figure(
    width=900, height=350,
    x_axis_type="datetime",
    title="Residuals through time",
    x_axis_label="Date",
    y_axis_label="Residual"
)
p_rt.line(df["dteday"], model_base.resid, line_width=1.5)
p_rt.scatter(df["dteday"], model_base.resid, size=4, alpha=0.45)
p_rt.add_layout(Span(location=0, dimension="width", line_dash="dashed"))
show(p_rt)

In [41]:
# Formal diagnostics.
bp = het_breuschpagan(model_base.resid, model_base.model.exog)
bg = acorr_breusch_godfrey(model_base, nlags=7)

diagnostic_tests = pd.DataFrame({
    "test": ["Breusch-Pagan heteroskedasticity", "Breusch-Godfrey autocorrelation (7 lags)"],
    "statistic": [bp[0], bg[0]],
    "p_value": [bp[1], bg[1]],
    "null_hypothesis": [
        "Constant conditional error variance",
        "No serial correlation through the tested lag order"
    ],
    "if_rejected": [
        "Classical OLS standard errors may be unreliable; consider robust SE / variance model",
        "Independence is doubtful; consider time structure, HAC SE, GLS/GLSAR"
    ]
})
show_bokeh_table(
    diagnostic_tests,
    title="Formal assumption diagnostics",
    explanation=("A diagnostic test is evidence about an assumption. It should guide investigation, "
                 "not replace graphical checks or subject-matter reasoning."),
    height=220
)

## 19. Leverage and influence

For observation $i$,

$$
h_{ii}=H_{ii}
$$

is its leverage.

High leverage means unusual predictor geometry.

A large residual means unusual response behavior.

An observation tends to be especially influential when it combines both.

Cook's distance summarizes how much the fitted model changes when an observation is perturbed/deleted.

In [60]:
p_lev = figure(
    width=900, height=380,
    title="Leverage vs externally studentized residual",
    x_axis_label="Leverage h_ii",
    y_axis_label="Externally studentized residual"
)
sizes = 5 + 40 * np.sqrt(np.clip(diag["cooks_d"], 0, diag["cooks_d"].quantile(0.99)))
p_lev.scatter(diag["leverage"], diag["studentized_external"], size=sizes, alpha=0.45)
p_lev.add_layout(Span(location=0, dimension="width", line_dash="dashed"))
show(p_lev)

### Diagnostic interpretation table

| Pattern | Possible problem | Consequence | Possible response |
|---|---|---|---|
| residual curvature | wrong mean functional form | biased conditional mean | transformation, polynomial/spline, interaction |
| funnel-shaped residuals | heteroskedasticity | inefficient OLS; wrong classical SEs | HC robust SEs, WLS/GLS, transform response |
| residual autocorrelation | dependent errors | wrong uncertainty estimates | time terms, HAC SEs, GLS/GLSAR |
| heavy Q-Q tails | non-Normal errors/outliers | exact small-sample inference weak | robust methods, bootstrap, transform |
| high leverage | unusual $X$ | estimate may depend strongly on point | inspect design/data quality |
| high Cook's distance | influential observation | coefficients may be unstable | sensitivity analysis |

### A useful reasoning pattern

- **Curvature** primarily threatens the mean specification.
- **Heteroskedasticity** primarily threatens classical efficiency/standard errors.
- **Autocorrelation** threatens independence-based uncertainty calculations.
- **Heavy tails / outliers** can threaten exact Normal-theory inference and stability.
- **Leverage** is about unusual predictor geometry, not necessarily a large residual.


# Part X — Robust inference and robust regression

## 20. OLS coefficients with heteroskedasticity-robust standard errors

If the conditional mean model is still acceptable but variance is not constant, we can preserve OLS point estimates while changing the covariance estimator.

This distinction matters:

- **OLS coefficient estimator** answers the mean-model problem;
- **standard-error estimator** quantifies uncertainty.

HC3 standard errors are a common finite-sample robust choice.

In [43]:
model_hc3 = model_base.get_robustcov_results(cov_type="HC3")

robust_se_compare = pd.DataFrame({
    "parameter": model_base.params.index,
    "OLS_estimate": model_base.params.values,
    "classical_SE": model_base.bse.values,
    "HC3_SE": model_hc3.bse,
    "SE_ratio_HC3_to_classical": model_hc3.bse / model_base.bse.values
})

robust_se_compare["student_message"] = np.where(
    robust_se_compare["SE_ratio_HC3_to_classical"] > 1.10,
    "HC3 materially increases estimated uncertainty.",
    np.where(robust_se_compare["SE_ratio_HC3_to_classical"] < 0.90,
             "HC3 materially decreases estimated uncertainty.",
             "Classical and HC3 uncertainty are fairly similar.")
)
show_bokeh_table(
    robust_se_compare,
    title="Classical vs HC3 standard errors",
    explanation="HC3 changes the covariance estimate, not the OLS point estimate.",
    height=430
)

## 21. Robust regression: change the loss, not just the SE

OLS minimizes

$$
\sum_i e_i^2,
$$

so large residuals receive quadratically increasing influence.

Huber-type robust regression behaves quadratically near zero but more gently in the tails.

This changes the **point estimate itself**, unlike HC3.

In [44]:
# Robust Linear Model using Huber's T loss.
# Use the same design matrix as the base OLS model.
rlm = sm.RLM(
    model_base.model.endog,
    model_base.model.exog,
    M=sm.robust.norms.HuberT()
).fit()

rlm_compare = pd.DataFrame({
    "parameter": model_base.params.index,
    "OLS": model_base.params.values,
    "Robust_Huber": rlm.params,
    "absolute_difference": np.abs(model_base.params.values - rlm.params)
})

rlm_view = rlm_compare.sort_values("absolute_difference", ascending=False)
rlm_view["student_message"] = "Large differences indicate sensitivity to tail/downweighting assumptions."
show_bokeh_table(
    rlm_view,
    title="OLS vs Huber robust regression",
    explanation="Unlike HC3, robust regression changes the fitting loss and therefore the point estimates.",
    height=420
)

# Part XI — Multiple comparisons

## 22. After ANOVA, which seasons differ?

Rejecting a global ANOVA null only says:

> not all group means are equal.

It does **not** tell us which pairs differ.

Testing every pair with ordinary 5% t-tests inflates family-wise Type-I error.

Tukey's HSD performs simultaneous pairwise comparisons with multiplicity control.

In [45]:
tukey = pairwise_tukeyhsd(
    endog=df["cnt"],
    groups=df["season_name"],
    alpha=0.05
)

print(tukey)

tukey_df = pd.DataFrame(
    tukey._results_table.data[1:],
    columns=tukey._results_table.data[0]
)
tukey_df["student_message"] = np.where(
    tukey_df["reject"].astype(bool),
    "Pairwise difference detected after Tukey multiplicity control.",
    "No pairwise difference detected after Tukey adjustment."
)
show_bokeh_table(
    tukey_df,
    title="Tukey HSD multiple comparisons",
    explanation="A global ANOVA rejection does not itself tell us which group pairs differ.",
    height=290
)

    Multiple Comparison of Means - Tukey HSD, FWER=0.05     
group1 group2  meandiff  p-adj    lower      upper    reject
------------------------------------------------------------
  Fall Spring -3040.1706    0.0 -3460.8003 -2619.5409   True
  Fall Summer  -651.9717 0.0004 -1070.8507  -233.0927   True
  Fall Winter  -916.1403    0.0  -1338.572  -493.7085   True
Spring Summer  2388.1989    0.0  1965.3325  2811.0653   True
Spring Winter  2124.0303    0.0  1697.6444  2550.4163   True
Summer Winter  -264.1686 0.3782  -688.8276   160.4904  False
------------------------------------------------------------


### Caution again

These are **unadjusted pairwise season means**.

If the scientific question is instead:

> which seasons differ after controlling for year, weather and temperature?

then we need contrasts or estimated marginal means from the multiple-regression model.

The target estimand must come before the test.

# Part XII — Generalized least squares and correlated errors

## 23. Why time ordering matters

The data are daily observations.

Even after modeling weather and trend, nearby days may have related errors.

Classical OLS assumes:

$$
\operatorname{Var}(\varepsilon)=\sigma^2I.
$$

A more general model allows:

$$
\operatorname{Var}(\varepsilon)=\sigma^2V.
$$

If $V$ is known,

$$
\hat\beta_{GLS}
=
(X^\top V^{-1}X)^{-1}X^\top V^{-1}Y.
$$

In practice $V$ is often estimated.  
`GLSAR` provides an iterative AR-error approximation.

In [46]:
# Fit GLSAR(1) on the same exogenous matrix as the base model.
# It is an extension for teaching, not a declaration that AR(1) is the true process.
glsar_model = sm.GLSAR(
    model_base.model.endog,
    model_base.model.exog,
    rho=1
)
glsar_result = glsar_model.iterative_fit(maxiter=12)

gls_compare = pd.DataFrame({
    "parameter": model_base.params.index,
    "OLS_coef": model_base.params.values,
    "GLSAR_coef": glsar_result.params,
    "OLS_SE": model_base.bse.values,
    "GLSAR_SE": glsar_result.bse
})

gls_compare["coef_difference"] = gls_compare["GLSAR_coef"] - gls_compare["OLS_coef"]
show_bokeh_table(
    gls_compare,
    title=f"OLS vs GLSAR | estimated AR parameter(s)={glsar_model.rho}",
    explanation=("GLS changes how correlated/unequal error covariance is handled. "
                 "The AR(1) specification is a teaching model, not an automatic truth."),
    height=420
)
print("Estimated AR coefficient(s):", glsar_model.rho)

Estimated AR coefficient(s): [0.49531278]


### OLS vs GLS: conceptual distinction

If the mean model is correct:

- OLS may remain unbiased under many forms of non-spherical errors;
- but it is no longer generally BLUE;
- classical OLS standard errors can be wrong;
- GLS can exploit covariance structure to improve efficiency.

This is where the Gauss–Markov theorem's assumptions become practically meaningful.

# Part XIII — Prediction is not the same objective as inference

## 24. Chronological train/test split

For explanatory inference, researchers often fit all observations relevant to the estimand.

For future prediction, evaluating on data used for fitting is misleading.

Because these observations are time ordered, we use an **earlier → later** split rather than a random split.

In [47]:
cut = int(len(df) * 0.80)
train_df = df.iloc[:cut].copy()
test_df = df.iloc[cut:].copy()

pred_model = smf.ols(formula_base, data=train_df).fit()
test_pred = pred_model.predict(test_df)

prediction_metrics = pd.DataFrame({
    "metric": ["MAE", "RMSE", "R2"],
    "value": [
        mean_absolute_error(test_df["cnt"], test_pred),
        np.sqrt(mean_squared_error(test_df["cnt"], test_pred)),
        r2_score(test_df["cnt"], test_pred)
    ],
    "interpretation": [
        "Typical absolute prediction error",
        "Penalizes large errors more strongly",
        "Fraction of holdout variance explained; can be negative"
    ]
})
show_bokeh_table(
    prediction_metrics,
    title="Chronological holdout performance",
    explanation=("Prediction asks how the fitted function generalizes to later data; "
                 "this is different from in-sample coefficient inference."),
    height=190
)

In [48]:
p_hold = figure(
    width=900, height=380,
    x_axis_type="datetime",
    title="Chronological holdout: actual vs predicted rentals",
    x_axis_label="Date",
    y_axis_label="Daily rentals"
)
p_hold.line(test_df["dteday"], test_df["cnt"], line_width=2, legend_label="Actual")
p_hold.line(test_df["dteday"], test_pred, line_width=2, legend_label="Predicted")
p_hold.legend.location = "top_left"
show(p_hold)

### Inference vs prediction

A variable can be:

- statistically significant but add little out-of-sample predictive value;
- predictively useful while its individual coefficient is unstable due to collinearity;
- associated with the response but unsuitable for causal interpretation.

These objectives overlap, but they are not identical.

# Part XIV — High-dimensional / regularized extension

## 25. What changes when the design matrix gets large?

Classical OLS becomes fragile when:

- $p$ approaches $n$;
- predictors are highly correlated;
- polynomial/interactions generate many columns.

Ridge regression solves:

$$
\min_\beta
\|Y-X\beta\|_2^2+\lambda\|\beta\|_2^2.
$$

Equivalent normal-equation form:

$$
\hat\beta_{\text{ridge}}
=
(X^\top X+\lambda I)^{-1}X^\top Y
$$

(up to intercept-handling conventions).

The bias introduced by $\lambda>0$ can reduce variance substantially.

In [49]:
# Prediction-oriented regularized model with richer feature generation.
numeric_features = ["temp_norm", "humidity_norm", "windspeed_norm", "time_index"]
categorical_features = ["season_name", "weather_name", "weekday_name", "yr", "workingday"]

numeric_pipe = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scale", StandardScaler())
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

ridge_pipe = Pipeline([
    ("prep", preprocess),
    ("ridge", RidgeCV(alphas=np.logspace(-3, 4, 60)))
])

ridge_pipe.fit(train_df[numeric_features + categorical_features], train_df["cnt"])
ridge_pred = ridge_pipe.predict(test_df[numeric_features + categorical_features])

ridge_metrics = pd.DataFrame({
    "model": ["Classical linear model", "Polynomial + one-hot + Ridge"],
    "MAE": [
        mean_absolute_error(test_df["cnt"], test_pred),
        mean_absolute_error(test_df["cnt"], ridge_pred)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(test_df["cnt"], test_pred)),
        np.sqrt(mean_squared_error(test_df["cnt"], ridge_pred))
    ],
    "R2": [
        r2_score(test_df["cnt"], test_pred),
        r2_score(test_df["cnt"], ridge_pred)
    ]
})

ridge_metrics["student_goal"] = [
    "Classical interpretable/inferential specification",
    "Variance-controlled predictive specification"
]
show_bokeh_table(
    ridge_metrics,
    title=f"Classical model vs Ridge | chosen alpha={ridge_pipe.named_steps['ridge'].alpha_:.4g}",
    explanation="Regularization trades bias for variance and changes the inferential framework.",
    height=200
)
print("Chosen ridge alpha:", ridge_pipe.named_steps["ridge"].alpha_)

Chosen ridge alpha: 0.3101168926574778


### What ridge does *not* give us automatically

Ridge is excellent for stabilizing prediction under collinearity, but its inferential interpretation differs from textbook OLS:

- coefficients are biased by design;
- ordinary OLS t-tests do not apply;
- feature scaling matters;
- cross-validation is commonly used to select $\lambda$.

This is an important bridge from ST4233 into statistical learning.

# Part XV — Generalized linear model extension

## 26. The response is actually a count

`cnt` is nonnegative integer-valued.

The Gaussian linear model treats the response as continuous with constant conditional variance.

A generalized linear model can instead specify:

$$
g(E[Y\mid X])=X\beta.
$$

For a Poisson model,

$$
\log E[Y\mid X]=X\beta.
$$

We fit a Poisson GLM only as a conceptual extension, then inspect dispersion.  
Count data with much larger variance than mean often exhibit **overdispersion**, making plain Poisson inference inadequate.

In [50]:
poisson_formula = (
    "cnt ~ temp_norm + humidity_norm + windspeed_norm "
    "+ C(yr) + C(season_name) + C(weather_name) + workingday + time_index"
)

poisson_model = smf.glm(
    poisson_formula,
    data=df,
    family=sm.families.Poisson()
).fit()

dispersion_ratio = poisson_model.pearson_chi2 / poisson_model.df_resid

glm_summary = pd.DataFrame({
    "quantity": ["Poisson deviance", "Pearson chi-square", "residual df", "Pearson dispersion ratio"],
    "value": [
        poisson_model.deviance,
        poisson_model.pearson_chi2,
        poisson_model.df_resid,
        dispersion_ratio
    ],
    "interpretation": [
        "Overall lack-of-fit measure for the Poisson GLM",
        "Alternative discrepancy measure",
        "Residual degrees of freedom",
        "Values far above 1 suggest overdispersion relative to Poisson"
    ]
})

show_bokeh_table(
    glm_summary,
    title="Poisson GLM dispersion audit",
    explanation=("A count response motivates a GLM, but Poisson mean-variance assumptions "
                 "must still be checked."),
    height=210
)

### General linear model vs generalized linear model

These names are easy to confuse.

**General linear model**

$$
Y=X\beta+\varepsilon
$$

with a continuous response and additive errors.

**Generalized linear model**

$$
g(E[Y\mid X])=X\beta
$$

with a distribution from the exponential family and a link function.

Examples include logistic and Poisson regression.

The common structural object is still the **linear predictor** $X\beta$.

# Part XVI — A compact end-to-end model audit

## 27. Build a decision table

A strong ST4233 analysis should not end with a coefficient table.

We ask, concept by concept:

- what did the mathematics assume?
- what did the data show?
- what should we do next?

In [51]:
bp_p = bp[1]
bg_p = bg[1]

audit = pd.DataFrame([
    ("Linearity", "E[Y|X] is linear in encoded predictors", "Inspect residual curvature / interactions", "Consider transformations/interactions if needed"),
    ("Full-rank design", "No exact redundant columns", f"Model rank={np.linalg.matrix_rank(model_base.model.exog)}; columns={model_base.model.exog.shape[1]}", "Avoid redundant encodings"),
    ("Collinearity", "Near-dependence should not be severe", f"temp-atemp correlation={corr_temp_atemp:.3f}", "Do not include both casually; use VIF/condition diagnostics"),
    ("Homoskedasticity", "Var(error|X) constant", f"Breusch-Pagan p={bp_p:.4g}", "Compare classical vs HC3 SE; consider WLS/GLS"),
    ("Independence", "Errors uncorrelated", f"Breusch-Godfrey p={bg_p:.4g}", "Time structure / HAC / GLSAR may be needed"),
    ("Normality", "Needed mainly for exact finite-sample t/F inference", "Inspect Q-Q plot", "Robust/asymptotic/bootstrap inference if tails problematic"),
    ("Influence", "No small set of rows dominates fit", f"max Cook's D={diag['cooks_d'].max():.4f}", "Inspect high-influence dates and sensitivity"),
    ("Prediction validity", "Future-like evaluation", f"Chronological holdout R²={r2_score(test_df['cnt'], test_pred):.3f}", "Prefer time-aware validation for forecasting claims"),
], columns=["topic", "assumption_or_goal", "evidence_in_notebook", "response"])

show_bokeh_table(
    audit,
    title="Final ST4233 model audit",
    explanation=("Connect each assumption to the evidence, the consequence of failure, "
                 "and an appropriate response."),
    height=340
)

# Part XVII — Computational complexity and numerical efficiency

The formulas in linear-model theory are compact, but implementation matters.

Let:

- $n$ = observations;
- $p$ = predictors.

| Operation | Typical cost | Comment |
|---|---:|---|
| form $X^\top X$ | $\Theta(np^2)$ | common normal-equation route |
| invert/solve $p\times p$ system | $\Theta(p^3)$ | explicit inverse usually avoidable |
| QR least squares | $\Theta(np^2)$ | numerically more stable |
| SVD least squares | $\Theta(np^2)$ for $n\gg p$ | strongest rank diagnostics, more expensive constants |
| form full hat matrix $H$ | $\Theta(n^2p)$ style cost / $O(n^2)$ memory | pedagogical only for large $n$ |
| predictions $X\hat\beta$ | $\Theta(np)$ | cheap after fitting |

### Efficiency tips

1. Prefer `np.linalg.lstsq`, QR, SVD, or a solver over explicitly forming $(X^\top X)^{-1}$.
2. Do not build an $n\times n$ hat matrix just to get leverage for large datasets.
3. Standardize predictors when using condition numbers or regularization.
4. Use sparse one-hot matrices when categorical expansion becomes wide.
5. Keep **inference matrices** and **prediction pipelines** conceptually separate.

# Part XVIII — Concept map: ST4233 theory ↔ notebook evidence

| ST4233 concept | Mathematical object | Where you saw it |
|---|---|---|
| linear model | $Y=X\beta+\varepsilon$ | simple and multiple demand models |
| OLS | $(X^\top X)^{-1}X^\top Y$ | manual matrix fit |
| projection | $H=X(X^\top X)^{-1}X^\top$ | symmetry/idempotence checks |
| residual maker | $M=I-H$ | residual projection |
| orthogonality | $X^\top e=0$ | numerical verification |
| ANOVA decomposition | $SST=SSR+SSE$ | direct calculation |
| covariance | $\sigma^2(X^\top X)^{-1}$ | manual vs statsmodels SE |
| Gaussian sampling | $\hat\beta\sim N(\beta,\cdots)$ | repeated-sample simulation |
| quadratic form | $Y^\top MY$ | SSE/chi-square simulation |
| Gauss–Markov | BLUE | OLS vs arbitrary WLS simulation |
| t/F inference | linear restrictions | coefficient tests and nested F-test |
| prediction | mean CI vs prediction interval | temperature profile |
| ANOVA | categorical design matrix | season model |
| interactions | product terms | temperature × season |
| collinearity | small singular/eigen values | temp + atemp, VIF, condition number |
| diagnostics | leverage, studentized residual, Cook's D | influence table/plots |
| multiple comparisons | simultaneous pairwise tests | Tukey HSD |
| GLS | $V\neq I$ | GLSAR extension |
| robust methods | robust covariance/loss | HC3 + Huber RLM |
| regularization | $X^\top X+\lambda I$ | Ridge extension |
| GLM | $g(E[Y|X])=X\beta$ | Poisson extension |

# Part XIX — Exercises for mastery

Try these without immediately looking up a solution.

### Exercise 1 — projection
For the simple temperature model, verify numerically that:

$$
\|Y\|^2=\|HY\|^2+\|MY\|^2
$$

only after handling the intercept/centering issue correctly. Explain the geometry.

### Exercise 2 — general linear hypothesis
Test:

$$
H_0:
\beta_{\text{humidity}}=
\beta_{\text{windspeed}}=0.
$$

Compare a direct `f_test` with a reduced-vs-full ANOVA comparison.

### Exercise 3 — centering
Fit the interaction model using uncentered temperature.  
Show that fitted values remain effectively unchanged while the interpretation of main effects changes.

### Exercise 4 — multicollinearity
Add both `temp` and `atemp` to the full model. Compare:

- fitted $R^2$;
- standard errors;
- VIF;
- condition number.

Explain why predictive fit may barely move while coefficient uncertainty changes sharply.

### Exercise 5 — heteroskedasticity
Compare confidence intervals using:

- classical OLS covariance;
- HC0;
- HC1;
- HC3.

Which coefficients' conclusions are most sensitive?

### Exercise 6 — influence
Remove the top 3 Cook's-distance observations, refit, and report the largest coefficient changes.  
Do not automatically delete influential points—first investigate them.

### Exercise 7 — GLSAR
Try AR orders 1, 3 and 7. Compare residual autocorrelation and coefficient uncertainty.  
Explain why choosing a covariance model solely because it lowers AIC can be dangerous.

### Exercise 8 — prediction
Replace the single 80/20 holdout with expanding-window `TimeSeriesSplit`.  
Compare OLS and Ridge RMSE across folds.

### Exercise 9 — GLM
Fit a Negative Binomial model and compare dispersion behavior with Poisson.  
Explain why count-valued $Y$ alone does not automatically invalidate OLS for a conditional-mean question.

### Exercise 10 — causal language
List which coefficients in this notebook can safely be described as **associations** and why none should automatically be called causal effects.

# Final synthesis

The main lesson of ST4233 is not "learn another regression package."

It is that a large family of statistical procedures is generated by a small number of mathematical ideas:

$$
\boxed{Y=X\beta+\varepsilon}
$$

$$
\boxed{\text{projection geometry}}
$$

$$
\boxed{\text{covariance propagation}}
$$

$$
\boxed{\text{quadratic forms}}
$$

$$
\boxed{\text{general linear hypotheses}}
$$

Once these are understood:

- regression and ANOVA become the same framework;
- t-tests and F-tests become structured comparisons of model-space directions;
- multicollinearity becomes a geometry/eigenvalue problem;
- leverage becomes a diagonal of a projection matrix;
- GLS becomes OLS after covariance-aware transformation;
- ridge becomes a controlled modification of an unstable normal equation.

The practical habit to carry forward is:

> **Specify the estimand → encode the design → fit the model → interrogate assumptions → quantify uncertainty → test sensitivity → distinguish inference from prediction.**

That is much closer to the spirit of advanced linear-model work than merely calling `.fit()`.

## References / provenance

### Dataset

**UCI Machine Learning Repository — Bike Sharing Dataset (id 275)**, donated by Hadi Fanaee-T (2013), DOI `10.24432/C5W894`.

UCI describes the data as hourly and daily Capital Bikeshare rental counts from 2011–2012 with weather and seasonal information. The dataset is intended for regression and includes `day.csv` and `hour.csv`.

This notebook uses the **daily file** and keeps the official normalized weather variables for modeling.

### Linear-model theory represented in the notebook

The progression intentionally mirrors advanced linear-model study:

- matrix algebra and rank;
- random vectors/covariance;
- multivariate Normal theory;
- quadratic forms;
- least squares and Gauss–Markov;
- general linear hypotheses;
- ANOVA and interactions;
- prediction and diagnostics;
- multicollinearity;
- multiple comparisons;
- GLS / robust extensions;
- regularization and GLM connections.

The exact assessed emphasis can vary by NUS semester/instructor; the current official ST4233 syllabus should remain the authority for examination scope.